In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade certifi pymongo

  Using cached certifi-2026.10.19-py3-none-any.whl.metadata (2.5 kB)
Using cached certifi-2026.10.19-py3-none-any.whl (136 kB)
  Attempting uninstall: certifi
    Found existing installation: certifi 2026.10.18
    Uninstalling certifi-2026.10.18:
      Successfully uninstalled certifi-2026.10.18
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pyspark
from pyspark.sql import SparkSession
from cassandra.cluster import Cluster
from pymongo import MongoClient
import certifi
import streamlit as st
import pandas as pd

In [4]:
# # Set up new keyspace (first time only)
# #                                              name of keyspace                        replication strategy           replication factor
# session.execute("CREATE KEYSPACE IF NOT EXISTS my_first_keyspace WITH REPLICATION = { 'class' : 'SimpleStrategy', 'replication_factor' : 1 };")

# # Create a new table (first time only)
# session.set_keyspace('my_first_keyspace')
# session.execute("DROP TABLE IF EXISTS my_first_keyspace.my_first_table;") # Starting from scratch every time
# session.execute("CREATE TABLE IF NOT EXISTS my_first_table (ind int PRIMARY KEY, company text, model text);")

# # Insert some data (ind is the primary key, must be unique)
# session.execute("INSERT INTO my_first_table (ind, company, model) VALUES (1, 'Tesla', 'Model Y');")
# session.execute("INSERT INTO my_first_table (ind, company, model) VALUES (2, 'Tesla', 'Model 3');")
# session.execute("INSERT INTO my_first_table (ind, company, model) VALUES (3, 'Polestar', '3');") 

# Connecting to Cassandra
cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()

session.set_keyspace("my_first_keyspace")

# testing cassandra works
rows = session.execute("SELECT * FROM my_first_table")
for row in rows:
    print(row)

In [5]:
#setting up spark session 
spark = SparkSession.builder \
    .appName("CassandraSparkTest") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"
    ) \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .getOrCreate()

26/10/20 12:51:46 WARN Utils: Your hostname, Mkken.local resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
26/10/20 12:51:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/vildeskobba/.ivy2/cache
The jars for the packages stored in: /Users/vildeskobba/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-653c8296-9abb-4f92-a40d-4785ebb7a06e;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/vildeskobba/Desktop/IND320/IND320_vilde/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3 in central
	found org.apache.cassandra#java-driver-mapper-runtime;4.18.1 in central
	found org.apache.cassandra#java-driver-query-builder;4.18.1 in central
	found org.apache.commons#commons-lang3;3.10 in central
	found com.thoughtworks.paranamer#paranamer;2.8 in central
	found org.scala-lang#scala-reflect

In [6]:
#testing spark works with cassandra 
df = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(keyspace="my_first_keyspace", table="my_first_table") \
    .load()

df.show()

+---+-------+-----+
|ind|company|model|
+---+-------+-----+
+---+-------+-----+



In [8]:
import os
import certifi
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv("/Users/vildeskobba/Desktop/IND320/IND320_vilde/.env", override=True)

uri = os.getenv("MONGO_URI")

print("URI loaded:", bool(uri))
print("Uses Atlas:", uri.startswith("mongodb+srv://") if uri else False)
print("CA bundle:", certifi.where())

URI loaded: True
Uses Atlas: True
CA bundle: /Users/vildeskobba/Desktop/IND320/IND320_vilde/.venv/lib/python3.13/site-packages/certifi/cacert.pem


In [7]:
client = MongoClient(
    st.secrets["MONGO_URI"],
    tlsCAFile=certifi.where()
)

print(client.admin.command("ping"))


26/10/20 12:52:08 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


ServerSelectionTimeoutError: SSL handshake failed: ac-t9esg44-shard-00-02.cymwxbj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1020) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-t9esg44-shard-00-00.cymwxbj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1020) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-t9esg44-shard-00-01.cymwxbj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1020) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6ad747cdcf5f67e63f386a4d, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-t9esg44-shard-00-00.cymwxbj.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: ac-t9esg44

In [ ]:
# test db operations with mongodb

db = client["test_database"]
collection = db["test_collection"]

# Insert
collection.insert_one({"item": "Tesla", "price": 10})

# Read
print(collection.find_one({"item": "Tesla"}))

# Update
collection.update_one(
    {"item": "Tesla"},
    {"$set": {"price": 20}}
)
print(collection.find_one({"item": "Tesla"}))

# Delete
collection.delete_one({"item": "Tesla"})
print(collection.find_one({"item": "Tesla"}))

{'_id': ObjectId('6ad67007fbde13c33e630218'), 'item': 'Tesla', 'price': 10}
{'_id': ObjectId('6ad67007fbde13c33e630218'), 'item': 'Tesla', 'price': 20}
None


In [ ]:
test_data = {
    "timestamp": [
        "2024-10-19 00:00:00+02:00",
        "2024-10-19 00:00:00+02:00",
        "2024-10-19 00:00:00+02:00",
        "2024-10-19 00:00:00+02:00",
        "2024-10-19 00:00:00+02:00",
    ],
    "from_zone": ["NO_1", "DE_LU", "NL", "NO_2", "NO_2"],
    "to_zone": ["SE_3", "NO_2", "NO_2", "NL", "GB"],
    "flow_MW": [0.00, 0.00, 0.00, 620.01, 0.00],
}

test_df = pd.DataFrame(test_data)
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])